# A-Star Search Algorithm

This notebook complements the walkthrough article [here](https://towardsdatascience.com/a-star-a-search-algorithm-eb495fb156bb) on the A-Star search algorithm. Throughout the notebook, we ask to you finish sections on your own and answer questions.

## [Question - Describe A*]
Describe A* search in your own words.

### Answer:

A* (A-Star) is a pathfinding algorithm used to find the shortest path between a starting point and a goal. It combines two ideas: the actual cost to reach a node from the start and an estimated cost to reach the goal from that node. The algorithm keeps track of nodes to explore in an "open list" and already visited nodes in a "closed list." At each step, it picks the node with the lowest total cost (actual + estimated) and explores its neighbors. This smart approach helps A* find the optimal path faster than simple search methods because it focuses on the most promising directions first. A* is widely used in video games for character navigation and in GPS systems for route planning.

## [Question - Describe h and g]
In A* search, describe the two key elements of *f(n)* (namely *h(n)* and *g(n)*). What do they represent in the algorithm and how do they differ?

### Answer:

In A* search, the evaluation function f(n) = g(n) + h(n) helps decide which node to explore next.

**g(n) - Actual Cost:**
- g(n) represents the actual cost of the path from the start node to the current node n.
- This value is exact and known because the algorithm has already traveled this path.
- It increases as we move further from the starting point.

**h(n) - Heuristic (Estimated Cost):**
- h(n) represents the estimated cost from the current node n to the goal node.
- This value is an approximation calculated using a heuristic function (like Euclidean or Manhattan distance).
- It guides the algorithm toward the goal by predicting which direction looks most promising.

**Key Difference:**
The main difference is that g(n) is a known, exact value based on the path already taken, while h(n) is an estimate or guess about the remaining distance. Together, they help A* balance between finding the shortest path (using g) and searching efficiently toward the goal (using h).

## [Activity - run Node]
Examine and run the below code.

In [17]:

class Node():
    """A node class for A* Pathfinding"""

    def __init__(self, parent=None, position=None):
        self.parent = parent
        self.position = position

        self.g = 0
        self.h = 0
        self.f = 0

    def __eq__(self, other):
        return self.position == other.position

## [Question - Explain 'Node']
In your own words, describe what the purpose of the *Node* class is. Why do we have to initialize a parent and a position?

### Answer:

The **Node class** serves as a building block for the A* algorithm. Each Node represents a single cell or location in the maze that the algorithm can visit.

**Purpose of Parent:**
The parent attribute stores a reference to the previous node in the path. This is essential because once we reach the goal, we need to trace back through all the parent nodes to reconstruct the complete path from start to end. Without tracking parents, we would know we reached the goal but wouldn't know how we got there.

**Purpose of Position:**
The position attribute stores the coordinates (row, column) of the node in the maze. This tells us exactly where this node is located in the grid. The algorithm uses position to check if a node is walkable, if it's within bounds, and to calculate distances.

The Node class also stores the g, h, and f values which are used to determine the cost of reaching this node and to compare it with other nodes when deciding which path to explore next.


## [Activity - Run astar() ]

In [18]:
def astar(maze, start, end):
    """Returns a list of tuples as a path from the given start to the given end in the given maze"""

    # Create start and end node
    start_node = Node(None, start)
    start_node.g = start_node.h = start_node.f = 0
    end_node = Node(None, end)
    end_node.g = end_node.h = end_node.f = 0

    # Initialize both open and closed list
    open_list = []
    closed_list = []

    # Add the start node
    open_list.append(start_node)

    # Loop until you find the end
    while len(open_list) > 0:

        # Get the current node
        current_node = open_list[0]
        current_index = 0
        for index, item in enumerate(open_list):
            if item.f < current_node.f:
                current_node = item
                current_index = index

        # Pop current off open list, add to closed list
        open_list.pop(current_index)
        closed_list.append(current_node)

        # Found the goal
        if current_node == end_node:
            path = []
            current = current_node
            while current is not None:
                path.append(current.position)
                current = current.parent
            return path[::-1] # Return reversed path

        # Generate children
        children = []
        for new_position in [(0, -1), (0, 1), (-1, 0), (1, 0), (-1, -1), (-1, 1), (1, -1), (1, 1)]: # Adjacent squares

            # Get node position
            node_position = (current_node.position[0] + new_position[0], current_node.position[1] + new_position[1])

            # Make sure within range
            if node_position[0] > (len(maze) - 1) or node_position[0] < 0 or node_position[1] > (len(maze[len(maze)-1]) -1) or node_position[1] < 0:
                continue

            # Make sure walkable terrain
            if maze[node_position[0]][node_position[1]] != 0:
                continue

            # Create new node
            new_node = Node(current_node, node_position)

            # Append
            children.append(new_node)

        # Loop through children
        for child in children:

            # Child is on the closed list
            if child in closed_list:
              continue

            # Create the f, g, and h values
            child.g = current_node.g + 1
            child.h = ((child.position[0] - end_node.position[0]) ** 2) + ((child.position[1] - end_node.position[1]) ** 2)
            child.f = child.g + child.h

            # Child is already in the open list
            for open_node in open_list:
                if child == open_node and child.g > open_node.g:
                    continue

            # Add the child to the open list
            open_list.append(child)


## [Question - Explain maze]
What type of input should value of *maze* be?

### Answer:

The **maze** input should be a **2D list (list of lists)** containing integer values.

- **0** represents walkable/open cells that the algorithm can pass through.
- **1** (or any non-zero value) represents walls or obstacles that block the path.

For example, a 3x3 maze would look like:
```python
maze = [
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0]
]
```

In this example, the center cell (1,1) is a wall, and all other cells are walkable. The algorithm accesses cells using `maze[row][column]` notation, so the maze must be a properly structured 2D array where each inner list represents a row of the grid.

## [Question - Build Main]
Please use the above code that uses the astar() function to find a path from the beginning to the end of a 7 by 7 maze. You can choose how the maze looks and where the start and end are.

In [19]:
# Define a 7x7 maze
# 0 = walkable path, 1 = wall/obstacle
maze = [
    [0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0]
]

# Define start and end positions
start = (0, 0)  # Top-left corner
end = (6, 6)    # Bottom-right corner

# Run the A* algorithm
path = astar(maze, start, end)

# Print results
print("Maze Layout (0 = open, 1 = wall):")
for row in maze:
    print(row)

print(f"\nStart position: {start}")
print(f"End position: {end}")
print(f"\nPath found:")
print(path)
print(f"\nPath length: {len(path)} steps")

Maze Layout (0 = open, 1 = wall):
[0, 0, 0, 1, 0, 0, 0]
[0, 1, 0, 1, 0, 1, 0]
[0, 1, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 0, 1, 0]
[0, 0, 0, 0, 0, 1, 0]
[1, 1, 1, 1, 0, 1, 0]
[0, 0, 0, 0, 0, 0, 0]

Start position: (0, 0)
End position: (6, 6)

Path found:
[(0, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 4), (5, 4), (6, 5), (6, 6)]

Path length: 9 steps


## [Question - Impossible Maze]
Now modify that maze so that the algorithm cannot solve it. What is the output of the algorithm?

In [20]:
## [Your Code Here]
# Define an impossible 7x7 maze where the end is blocked
# Adding walls to completely block the path to the end
impossible_maze = [
    [0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 0, 1, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 1],  # Added wall at (4,6)
    [1, 1, 1, 1, 0, 1, 1],  # Added wall at (5,6)
    [0, 0, 0, 0, 0, 1, 0]   # Wall at (6,5) blocks access to end
]

# Define start and end positions
start = (0, 0)  # Top-left corner
end = (6, 6)    # Bottom-right corner (now blocked)

# Run the A* algorithm
path = astar(impossible_maze, start, end)

# Print results
print("Impossible Maze Layout (0 = open, 1 = wall):")
for row in impossible_maze:
    print(row)

print(f"\nStart position: {start}")
print(f"End position: {end}")
print(f"\nPath found: {path}")

print("\nExplanation: The algorithm returns 'None' because there is no valid path")
print("from the start to the end. The end position (6,6) is completely blocked")
print("by walls, making it impossible to reach.")

Impossible Maze Layout (0 = open, 1 = wall):
[0, 0, 0, 1, 0, 0, 0]
[0, 1, 0, 1, 0, 1, 0]
[0, 1, 0, 0, 0, 1, 0]
[0, 1, 1, 1, 0, 1, 0]
[0, 0, 0, 0, 0, 1, 1]
[1, 1, 1, 1, 0, 1, 1]
[0, 0, 0, 0, 0, 1, 0]

Start position: (0, 0)
End position: (6, 6)

Path found: None

Explanation: The algorithm returns 'None' because there is no valid path
from the start to the end. The end position (6,6) is completely blocked
by walls, making it impossible to reach.
